# IMU - Thesis

Filip and Gottfrid

## 1.2 Reading Data from Files

Read your data from the file into a DataFrame variable so that the column names correspond with
the field names from the dataset. Print a list of all the column names.

First we start of by importing all libraries needed for this assignment and defining paths for the files.

### 1.2.1 Importing libraries

In [1]:
#@title
# Import all libraries needed for this tutorial

# Pandas: used to create and work with DataFrames
import pandas as pd #imports all of pandas library
from pandas import DataFrame, read_csv #imports some of the important functions from the pandas library.

# Scikit-learn: import preprocessing functions
from sklearn.preprocessing import LabelEncoder, label_binarize, OrdinalEncoder

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Matplotlib: used for data visualization/plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
# Enable inline plotting, allows the plots to be seen in the notebook and stored with the document
%matplotlib inline

# numpy
import numpy as np

# seaborn: used for heatmaps and other statistical plots
%pip install seaborn
import seaborn as sns #might require installation with: conda install -c anaconda seaborn

# Importing statistical function for t-test from scipy
import scipy.stats as stats

# import KMeans
from sklearn.cluster import KMeans
# import nearest neighbors
#from sklearn.neighbors import NearestNeighbors

# Saving variables as files on disk
import pickle

import warnings
warnings.filterwarnings("ignore") #some warnings are supressed

#jupyter notebook --no-browser

Note: you may need to restart the kernel to use updated packages.


#### 1.2.2.2 Using own computer

In [2]:
# If you are working on your own computer, change the working directory to the lab-1 folder
import os

# Print the current working directory
print("Current working directory:", os.getcwd())


Current working directory: /Users/filipnordqvist/Documents/GitHub/Examensarbete


In [3]:
#print("Current working directory:", os.getcwd())

# Opening and saving data to variables

# Saving raw data
file_path = 'data/metadata_tracks.csv' #path to the file
rawdata = pd.read_csv(file_path, sep=",", encoding='latin1')

# Set display options to show all columns
pd.set_option('display.max_columns', None)

# Display the entire dataset
print(rawdata)


     subject  testID testName  isPatient  distanceReference  hasMotion  \
0          0       0      0_0       True                623      False   
1          0       1      0_1       True                598      False   
2          0       2      0_2       True                584      False   
3          0       3      0_3       True                577      False   
4          0       4      0_4       True                590      False   
..       ...     ...      ...        ...                ...        ...   
198       18       0     18_0      False                317       True   
199       18       1     18_1      False                365       True   
200       18       2     18_2      False                452       True   
201       18       3     18_3      False                461       True   
202       18       4     18_4      False                517       True   

     hasGNSS            device  distanceByApp  totSteps  path curvature  \
0       True  Apple iPhone11,8     5

### Looking at the data


In [4]:
rawdata.head()

,subject,testID,testName,isPatient,distanceReference,hasMotion,hasGNSS,device,distanceByApp,totSteps,path curvature,total_gaps_time_inertial,total_gaps_time_gnss,gt_type,country,gnss_anonimized,duration [s],fs_acc,fs_gnss,fs_steps,average_walking_speed,smartphone_position,smartphone app
0,0,0,0_0,True,623,False,True,"Apple iPhone11,8",598.373421,755.0,1,NaN,0.0,final,UK,True,368,NaN,0.304348,0.380435,1.692935,hand held,Timed Walk
1,0,1,0_1,True,598,False,True,"Apple iPhone11,8",587.334804,699.0,1,NaN,0.0,final,UK,True,380,NaN,0.292105,0.357895,1.573684,hand held,Timed Walk
2,0,2,0_2,True,584,False,True,"Apple iPhone11,8",577.005522,726.0,1,NaN,0.0,final,UK,True,359,NaN,0.281337,0.381616,1.626741,hand held,Timed Walk
3,0,3,0_3,True,577,False,True,"Apple iPhone11,8",539.365038,710.0,1,NaN,0.0,final,UK,True,365,NaN,0.284932,0.372603,1.580822,hand held,Timed Walk
4,0,4,0_4,True,590,False,True,"Apple iPhone11,8",566.744323,715.0,1,NaN,0.0,final,UK,True,360,NaN,0.277778,0.375000,1.638889,hand held,Timed Walk


In [5]:
rawdata.dtypes

subject                       int64
testID                        int64
testName                     object
isPatient                      bool
distanceReference             int64
hasMotion                      bool
hasGNSS                        bool
device                       object
distanceByApp               float64
totSteps                    float64
path curvature                int64
total_gaps_time_inertial    float64
total_gaps_time_gnss        float64
gt_type                      object
country                      object
gnss_anonimized                bool
duration [s]                  int64
fs_acc                      float64
fs_gnss                     float64
fs_steps                    float64
average_walking_speed       float64
smartphone_position          object
smartphone app               object
dtype: object

In [6]:
rows, columns = rawdata.shape
print(f"The dataset has {rows} rows and {columns} columns.")

The dataset has 203 rows and 23 columns.


In [7]:
#@First and last phone used in the dataset
#Examples of addressing cells
print(rawdata.iloc[0,7], rawdata.iloc[202,7])

Apple iPhone11,8 OPPO CPH2247


In [8]:
# Get the unique smartphone positions
unique_positions = rawdata['smartphone_position'].unique().tolist()

# Count the number of unique positions
num_unique_positions = len(unique_positions)

print(f'There are {num_unique_positions} different types of smartphone positions.')

There are 1 different types of smartphone positions.


In [9]:
# Count the occurrences of True and False in the 'isPatient' column
patient_counts = rawdata['isPatient'].value_counts()

# Summary
summary = {
    'Is Patient': patient_counts[True],
    'Not Patient': patient_counts[False]
}

print(summary)

{'Is Patient': 75, 'Not Patient': 128}


In [10]:
rawdata['country'].value_counts()

country
UK    106
SE     97
Name: count, dtype: int64

In [11]:
#@title
# Example: Listing all column names of rawdata. Observe the spaces in several of the names!
rawdata.columns

Index(['subject', 'testID', 'testName', 'isPatient', 'distanceReference',
       'hasMotion', 'hasGNSS', 'device', 'distanceByApp', 'totSteps',
       'path curvature', 'total_gaps_time_inertial', 'total_gaps_time_gnss',
       'gt_type', 'country', 'gnss_anonimized', 'duration [s]', 'fs_acc',
       'fs_gnss', 'fs_steps', 'average_walking_speed', 'smartphone_position',
       'smartphone app'],
      dtype='object')

In [12]:
#No swedish patients in the dataset
#rawdata[(rawdata['hasMotion'] == True) & (rawdata['hasGNSS'] == True) & (rawdata['subject'] == '3')]
rawdata[(rawdata['subject'] == 3)]

,subject,testID,testName,isPatient,distanceReference,hasMotion,hasGNSS,device,distanceByApp,totSteps,path curvature,total_gaps_time_inertial,total_gaps_time_gnss,gt_type,country,gnss_anonimized,duration [s],fs_acc,fs_gnss,fs_steps,average_walking_speed,smartphone_position,smartphone app
35,3,0,3_0,True,537,False,True,"Apple iPhone13,2",535.114992,642.0,1,NaN,0.0,final,UK,True,361,NaN,0.265928,0.379501,1.487535,hand held,Timed Walk
36,3,1,3_1,True,521,True,True,OnePlus KB2003,550.054111,662.0,1,12.0,16.0,final,UK,True,362,57.284530,0.969613,1.707182,1.439227,hand held,Timed Walk
37,3,2,3_2,True,522,True,True,OnePlus KB2003,514.496168,650.0,0,23.0,26.0,final,UK,True,360,55.969444,0.947222,1.713889,1.450000,hand held,Timed Walk
38,3,3,3_3,True,527,True,True,OnePlus KB2003,538.652157,656.0,1,0.0,0.0,final,UK,True,361,59.670360,1.573407,1.770083,1.459834,hand held,Timed Walk
39,3,4,3_4,True,561,True,True,OnePlus KB2003,574.514709,682.0,1,0.0,0.0,final,UK,True,364,59.255495,1.019231,1.857143,1.541209,hand held,Timed Walk
40,3,5,3_5,True,495,False,True,"Apple iPhone13,2",189.523236,579.0,2,NaN,172.0,final,UK,True,354,NaN,0.180791,0.381356,1.398305,hand held,Timed Walk
41,3,6,3_6,True,523,False,True,"Apple iPhone13,2",316.507008,630.0,2,NaN,73.0,final,UK,True,359,NaN,0.225627,0.378830,1.456825,hand held,Timed Walk
42,3,7,3_7,True,539,False,True,"Apple iPhone13,2",341.602504,621.0,2,NaN,67.0,final,UK,True,356,NaN,0.250000,0.382022,1.514045,hand held,Timed Walk
43,3,8,3_8,True,541,False,True,"Apple iPhone13,2",160.943639,636.0,2,NaN,74.0,final,UK,True,177,NaN,0.197740,0.802260,3.056497,hand held,Timed Walk
44,3,9,3_9,True,568,False,True,"Apple iPhone13,2",270.567258,660.0,2,NaN,82.0,final,UK,True,366,NaN,0.229508,0.382514,1.551913,hand held,Timed Walk


In [13]:
# Accuracy of the GNSS data with IMU data

GPS_IMU = rawdata[(rawdata['hasMotion'] == True) & (rawdata['hasGNSS'] == True) & (rawdata['subject'] == 3)]
rawdata['distance_with_GPS_IMU'] = rawdata['average_walking_speed'] * rawdata['duration [s]']




In [14]:
GPS = rawdata[(rawdata['hasMotion'] == False) & (rawdata['hasGNSS'] == True) & (rawdata['subject'] == 3)]
print(GPS)


    subject  testID testName  isPatient  distanceReference  hasMotion  \
35        3       0      3_0       True                537      False   
40        3       5      3_5       True                495      False   
41        3       6      3_6       True                523      False   
42        3       7      3_7       True                539      False   
43        3       8      3_8       True                541      False   
44        3       9      3_9       True                568      False   

    hasGNSS            device  distanceByApp  totSteps  path curvature  \
35     True  Apple iPhone13,2     535.114992     642.0               1   
40     True  Apple iPhone13,2     189.523236     579.0               2   
41     True  Apple iPhone13,2     316.507008     630.0               2   
42     True  Apple iPhone13,2     341.602504     621.0               2   
43     True  Apple iPhone13,2     160.943639     636.0               2   
44     True  Apple iPhone13,2     270.567258

In [15]:
#@title
# Example: Finding missing values in the original data
rawdata.isnull().sum()

subject                      0
testID                       0
testName                     0
isPatient                    0
distanceReference            0
hasMotion                    0
hasGNSS                      0
device                       0
distanceByApp                5
totSteps                     5
path curvature               0
total_gaps_time_inertial    52
total_gaps_time_gnss         5
gt_type                      0
country                      0
gnss_anonimized              0
duration [s]                 0
fs_acc                      52
fs_gnss                      5
fs_steps                     5
average_walking_speed        0
smartphone_position          0
smartphone app               0
distance_with_GPS_IMU        0
dtype: int64

In [16]:
Remove = ['subject', 'testID', 'testName', 'isPatient','device', 'path curvature','gt_type', 'country', 'smartphone_position',	'smartphone app' ]
clean_data = rawdata.drop(columns=Remove)


In [17]:
clean_data.head()

,distanceReference,hasMotion,hasGNSS,distanceByApp,totSteps,total_gaps_time_inertial,total_gaps_time_gnss,gnss_anonimized,duration [s],fs_acc,fs_gnss,fs_steps,average_walking_speed,distance_with_GPS_IMU
0,623,False,True,598.373421,755.0,NaN,0.0,True,368,NaN,0.304348,0.380435,1.692935,623.0
1,598,False,True,587.334804,699.0,NaN,0.0,True,380,NaN,0.292105,0.357895,1.573684,598.0
2,584,False,True,577.005522,726.0,NaN,0.0,True,359,NaN,0.281337,0.381616,1.626741,584.0
3,577,False,True,539.365038,710.0,NaN,0.0,True,365,NaN,0.284932,0.372603,1.580822,577.0
4,590,False,True,566.744323,715.0,NaN,0.0,True,360,NaN,0.277778,0.375000,1.638889,590.0


In [18]:
true_false_counts = rawdata['total_gaps_time_gnss'].value_counts()
print(true_false_counts)

total_gaps_time_gnss
0.0      129
21.0       5
9.0        4
7.0        4
15.0       4
1.0        3
43.0       2
33.0       2
17.0       2
25.0       1
195.0      1
144.0      1
235.0      1
44.0       1
380.0      1
216.0      1
10.0       1
49.0       1
23.0       1
145.0      1
130.0      1
122.0      1
68.0       1
59.0       1
56.0       1
128.0      1
168.0      1
198.0      1
93.0       1
8.0        1
2.0        1
121.0      1
85.0       1
70.0       1
40.0       1
102.0      1
63.0       1
57.0       1
45.0       1
41.0       1
34.0       1
31.0       1
22.0       1
50.0       1
16.0       1
26.0       1
172.0      1
73.0       1
67.0       1
74.0       1
82.0       1
95.0       1
Name: count, dtype: int64
